In [72]:
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pickle
import numpy as np
import bloom


In [73]:
df = pd.read_csv("dataset/urldata.csv")
len(df)

420464

In [74]:
df['label'] = df['label'].map({'good':0,'bad':1})
good_urls = df[df['label']==0]['url'].tolist()
bad_urls = df[df['label']==1]['url'].tolist()

In [75]:
train_df,test_df = train_test_split(df,test_size=0.3,random_state=42)

In [76]:
vectorizer = HashingVectorizer(n_features=2**10,alternate_sign=False)
X_train = vectorizer.fit_transform(train_df['url'])
X_test = vectorizer.transform(test_df['url'])
y_train = train_df['label']
y_test = test_df['label']

In [77]:
model = LogisticRegression()
model.fit(X_train,y_train)

LogisticRegression()

In [78]:
pickle.dump(model,open("model.pkl","wb"))
pickle.dump(vectorizer,open("vectorizer.pkl","wb"))

preparing url for standard bf

In [79]:
# with open("good_urls.txt","w") as f:
#     for url in good_urls:
#         f.write(url + "\n")

In [80]:
# with open("bad_urls.txt", "w", encoding="utf-8") as f:
#     for url in df[df['label'] == 0]['url']:
#         f.write(url + "\n")

preparing fn list for learned bf

In [81]:
probs = model.predict_proba(X_train)[:, 1]

In [82]:
threshold = 0.6
preds = (probs>=threshold).astype(int)

In [83]:
fn_mask = (y_train==1) & (preds==0)

In [84]:
fn_urls = train_df['url'][fn_mask].tolist()

In [85]:
print("Total bad (positives):", np.sum(y_train == 1))
print("Total good (negatives):", np.sum(y_train == 0))
fn_rate = len(fn_urls) / np.sum(y_train == 1)
print("FN rate:", fn_rate)

Total bad (positives): 52923
Total good (negatives): 241401
FN rate: 0.44190616556128715


In [86]:
false_positives = np.sum((y_train == 0) & (probs >= threshold))
total_good = np.sum(y_train == 0)

model_fpr = false_positives / total_good
print("Model FPR:", model_fpr)

Model FPR: 0.016346245458800916


In [87]:
fn_count = len(fn_urls)
fn_count

23387

In [88]:
learned_bf = bloom.BloomFilter(n = fn_count,fpr=0.1)
learned_bf.insert_batch(fn_urls)

In [89]:
test_urls = test_df['url']
test_labels = test_df['label']
test_probs = model.predict_proba(X_test)[:,1]

In [90]:
model_preds = [p>=threshold for p in test_probs]
bf_toquery = [url for url,mp in zip(test_urls,model_preds) if not mp]

In [91]:
bf_output = learned_bf.query_batch(bf_toquery)

In [92]:
bf_idx = 0
result = []
for mp in model_preds:
    if(mp): result.append(True)
    else:
        result.append(bool(bf_output[bf_idx]))
        bf_idx +=1


In [93]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(test_labels, result))

[[93194 10226]
 [ 7794 14926]]


In [94]:
falsePos = 0
falseNeg = 0
totalBad = 0
totalGood = 0

for res, label in zip(result, test_labels):
    if label == 1:   # bad
        totalBad += 1
        if not res:  # predicted good → FN
            falseNeg += 1
    else:            # good
        totalGood += 1
        if res:      # predicted bad → FP
            falsePos += 1

learned_system_fpr = falsePos / totalGood
learned_system_fnr = falseNeg / totalBad

print("System FPR:", learned_system_fpr)
#print("System FNR:", learned_system_fnr)

System FPR: 0.09887836008508992


standard bloom filter with same fpr storing minority

In [95]:
train_bad_urls = train_df.loc[train_df['label'] == 1, 'url'].tolist()

In [96]:
target_fpr = learned_system_fpr
n = len(train_bad_urls)
n

52923

In [97]:
std_bf = bloom.BloomFilter(n=n, fpr=0.12)
std_bf.insert_batch(train_bad_urls)

In [98]:
test_urls = test_df["url"].tolist()

In [99]:
std_results = std_bf.query_batch(test_urls)

In [100]:
falsePos = 0
totalGood = 0

for res, label in zip(std_results, test_labels):
    if label == 0:   # good
        totalGood += 1
        if res:      # predicted bad
            falsePos += 1

std_fpr = falsePos / totalGood
print("Standard BF FPR:", std_fpr)

Standard BF FPR: 0.10002900792883389


evaculation loop

In [101]:
import os

model_size = os.path.getsize("model.pkl") / (1024 * 1024)
vectorizer_size = os.path.getsize("vectorizer.pkl") / (1024 * 1024)
learned_bf_size = learned_bf.getMemoryBits()/(8*1024*1024)
print(model_size)
print(vectorizer_size)
print(learned_bf_size)

print(f"Learned System: {model_size+vectorizer_size+learned_bf_size} MB")


0.0084991455078125
0.000396728515625
0.01336669921875
Learned System: 0.0222625732421875 MB


In [102]:
std_bf_size = std_bf.getMemoryBits() / (8 * 1024 * 1024)
print("Standard BF size (MB):", std_bf_size)

Standard BF size (MB): 0.0278472900390625
